In [1]:
# Cell 1 — imports (same as original notebook)
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.base import clone
from sklearn.metrics import (accuracy_score, precision_score,
                             recall_score, confusion_matrix, f1_score)
import warnings
warnings.filterwarnings('ignore')

RANDOM_STATE = 42
DATA_PATH = Path("../data/power_system_combined.csv")

In [2]:
df = pd.read_csv(DATA_PATH)
print("DataFrame shape:", df.shape)

DataFrame shape: (78377, 130)


## 3. Column decode

Understanding what each of the columns mean for feature importance interpretability

| Token | Meaning |
|---|---|
| `R1`–`R4` | Which relay the measurement comes from |
| `PA` | **P**hasor **A**ngle (degrees) |
| `PM` | **P**hasor **M**agnitude |
| `:V` / `:VH` | Voltage magnitude / voltage angle |
| `:I` / `:IH` | Current magnitude / current angle |
| `R#:F` | Frequency (Hz, nominal 60) |
| `R#:DF` | Rate of change of frequency (df/dt) |
| `R#-PA:Z` | **Apparent impedance magnitude** (Z = V / I) -> Ohm's Law |
| `R#-PA:ZH` | Apparent impedance angle |
| `R#:S` | Relay status word (bitfield — *not* a PMU measurement) |

### The 12 phasor pairs per relay

Each relay reports 12 phasor pairs, in a fixed order that follows standard power-engineering
fault analysis:

| Pairs | Quantity |
|---|---|
| 1–3 | Three-phase **voltage** (A, B, C) |
| 4–6 | Three-phase **current** (A, B, C) |
| 7–9 | **Symmetrical-component voltage** (positive, negative, zero sequence) |
| 10–12 | **Symmetrical-component current** (positive, negative, zero sequence) |

The symmetrical-component decomposition is the classical tool for distinguishing balanced from
unbalanced faults: a healthy three-phase system has near-zero negative- and zero-sequence
components, so non-zero values there indicate an asymmetric disturbance.

### `R#-PA:Z` is a critical column

Distance (impedance) protective relays trip on **apparent impedance**. Under normal load the
line looks like a large impedance; during a fault, current spikes and Z collapses toward the
origin of the R–X plane, crossing the relay's trip characteristic. A false-data-injection attack
that wants to cause a spurious trip must therefore spoof exactly this signal. Prior published
work on this dataset independently ranks impedance and phase angle as the most diagnostic
features, so `PA:Z` is both the physically motivated and the empirically expected winner.

### The cyber-layer log columns

`control_panel_log1-4`, `relay1_log`–`relay4_log`, `snort_log1-4` - 12 columns from the control
panel, relay logs, and the Snort intrusion-detection system. These are network/cyber-layer, not
PMU-derived. Section 6 tests empirically whether they carry any signal.

In [3]:
#Group contiguous runs of the same label into blocks. 
#A block is a contiguous run of rows with the same label and source file. 
#The block_id is a unique identifier for each block.
#This is a necessary step, as the data is strongly autocorrelated, 
#and a random split would put near-duplicate rows in both train and test sets, 
#inflating performance metrics.
change = ((df["marker"] != df["marker"].shift())
          | (df["source_file"] != df["source_file"].shift()))
df["block_id"] = change.cumsum()

blocks = df.groupby("block_id").agg(rows=("marker", "size"),
                                    label=("marker", "first"),
                                    n_labels=("marker", "nunique"),
                                    src=("source_file", "first"))

assert df["block_id"].nunique() == 75, f"expected 75 blocks, got {df['block_id'].nunique()}"
assert (blocks["n_labels"] == 1).all(), "a block spans more than one label"
assert (blocks.groupby("src").size() == 5).all(), "every source file should hold exactly 5 blocks"

adj = (df["marker"].values[1:] == df["marker"].values[:-1]).mean()
print(f"{len(blocks)} contiguous label-pure blocks  (15 files x 5 runs)")
print(f"rows per block: min={blocks.rows.min()}  median={int(blocks.rows.median())}  max={blocks.rows.max()}")
print(f"adjacent rows sharing a label: {adj:.4f}")

# Every file has the SAME N/A/N/A/N layout, so a "block" is not one event. It is
# ~100+ events of one class concatenated. Split blocks at large discontinuities to
# recover an approximate per-event grouping.
v = df["R1-PM1:V"].to_numpy()
jump = np.r_[True, np.abs(np.diff(v)) > 10_000]
df["event_id"] = (change.to_numpy() | jump).cumsum()

GROUPINGS = {
    #15 original source files -> 15 groups
    "source_file": df["source_file"],
    #75 contiguous label runs -> 75 groups
    "block": df["block_id"],           
    #~2.6k approximate individual events -> 2.6k groups
    "event": df["event_id"],
}
for name, gr in GROUPINGS.items():
    print(f"grouping '{name}': {gr.nunique()} groups")

print("\n-> Rows are strongly autocorrelated. A random split puts near-duplicate")
print("   neighbours in both train and test. ")

75 contiguous label-pure blocks  (15 files x 5 runs)
rows per block: min=79  median=619  max=3537
adjacent rows sharing a label: 0.9992
grouping 'source_file': 15 groups
grouping 'block': 75 groups
grouping 'event': 2625 groups

-> Rows are strongly autocorrelated. A random split puts near-duplicate
   neighbours in both train and test. 


In [4]:
RELAYS = [1, 2, 3, 4]

GENERIC_PHASOR = []
for i in range(1, 13):
    kind = "V" if (i <= 3 or 7 <= i <= 9) else "I"
    GENERIC_PHASOR += [f"PA{i}:{kind}H", f"PM{i}:{kind}"]

GENERIC_29 = GENERIC_PHASOR + ["F", "DF", "PA:Z", "PA:ZH", "S"]
GENERIC_PMU_28 = [g for g in GENERIC_29 if g != "S"]

def wide(relay: int, generic: str) -> str:
    #Map a relay number + generic feature name onto its wide column name.
    return f"R{relay}:{generic}" if generic in ("F", "DF", "S") else f"R{relay}-{generic}"
    
def to_long(frame: pd.DataFrame, generic_feats=GENERIC_PMU_28) -> pd.DataFrame:
    #Wide (one row = one system snapshot) -> long (one row = one relay's view).

    #Feature columns are renamed to relay-generic names, so relay identity lives in
    #the relay_id column rather than being baked into column names. block_id and
    #orig_row are carried through so grouped CV and leave-one-relay-out can both
    #respect the original row structure.
    
    parts = []
    for r in RELAYS:
        rename = {wide(r, g): g for g in generic_feats}
        part = frame[list(rename)].rename(columns=rename)
        part["relay_id"] = r
        part["marker"] = frame["marker"].to_numpy()
        part["source_file"] = frame["source_file"].to_numpy()
        part["block_id"] = frame["block_id"].to_numpy()
        part["event_id"] = frame["event_id"].to_numpy()
        part["orig_row"] = frame.index.to_numpy()
        parts.append(part)
    return pd.concat(parts, ignore_index=True)

MINIMAL = ['PM2:V', 'PM1:V', 'PM5:I', 'F', 'PA3:VH', 'PM4:I', 'PA2:VH', 'PA:ZH']

long_df = to_long(df)
X_long = long_df[GENERIC_PMU_28].copy()
y_long = (long_df["marker"] == "Attack").astype(int)
g_long = long_df["source_file"]
Z_LONG = []

#**Note for the Above**: Z_LONG is intentionally empty because none of 8 features in the minimal set deal with infs

In [5]:
def fit_inf_policy(X_train: pd.DataFrame, z_cols, quantile: float = 0.999) -> dict:
    #Compute the cap as the given quantile of that column's FINITE values in X_train
    #Fit on training only so that the test fold doesn't influence how the training fold is transformed
    policy = {}
    for c in z_cols:
        col = X_train[c]
        finite = col[np.isfinite(col)]          # the critical guard
        policy[c] = float(finite.quantile(quantile))
    return policy

def apply_inf_policy(X: pd.DataFrame, policy: dict, z_cols) -> pd.DataFrame:
    #Implement the flag and cap policy.
    #New DataFrame is returned so that the original X isn't mutated.
    out = X.copy()                              
    for c in z_cols:
        cap = policy[c]
        out[f"{c}__open"] = np.isposinf(out[c]).astype(int)
        #Flag before capping, so that the model can learn that a breaker opened
        #and then the voltage/current collapsed to zero.   
        out[c] = out[c].clip(upper=cap)
    return out

#Evaluation harness. Both protocols refit the inf policy inside every fold
def _metrics(y_true, y_pred) -> dict:
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "macro_f1": f1_score(y_true, y_pred, average="macro"),
    }

def eval_grouped(model, X, y, groups, z_cols, n_splits=5):
    #StatifiedGroupKFold over contiguous label-run blocks to test the honest protocol
    #Each row is predicted on a model that never saw its block during training
    #Out-of-fold metrics are returned.
    cv = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_STATE)
    groups = pd.Series(groups).reset_index(drop=True)
    oof = np.empty(len(y), dtype=int)
    for tr_idx, te_idx in cv.split(X, y, groups):
        # the wiring check: no group may appear on both sides of the split
        assert not set(groups.iloc[tr_idx]) & set(groups.iloc[te_idx]), "group leak"
        X_tr, X_te = X.iloc[tr_idx], X.iloc[te_idx]
        policy = fit_inf_policy(X_tr, z_cols)
        fitted = clone(model).fit(apply_inf_policy(X_tr, policy, z_cols), y.iloc[tr_idx])
        oof[te_idx] = fitted.predict(apply_inf_policy(X_te, policy, z_cols))
    return _metrics(y, oof), confusion_matrix(y, oof)

In [6]:
# Files committed to holdout
TEST_FILES = ["data6.csv", "data11.csv", "data15.csv"]

test_mask = long_df["source_file"].isin(TEST_FILES)
long_train = long_df[~test_mask].copy().reset_index(drop=True)
long_test  = long_df[test_mask].copy().reset_index(drop=True)

X_train = long_train[MINIMAL]
y_train = (long_train["marker"] == "Attack").astype(int)
g_train = long_train["source_file"]

X_test  = long_test[MINIMAL]
y_test  = (long_test["marker"] == "Attack").astype(int)

In [7]:
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

lr_8 = Pipeline([
    ("scale", StandardScaler()),
    ("clf", LogisticRegression(max_iter=2000, class_weight="balanced",
                               random_state=RANDOM_STATE)),
])

m, _ = eval_grouped(lr_8, X_train, y_train, g_train, [])
print(f"LR balanced, long format, 8 features: macro_f1={m['macro_f1']:.4f}  acc={m['accuracy']:.4f}")
print(f"Floor (always-predict-Attack): 0.4168")

LR balanced, long format, 8 features: macro_f1=0.4732  acc=0.4883
Floor (always-predict-Attack): 0.4168


In [8]:
import xgboost as xgb
#Test a model config that is feasible to implement on hardware
xgb_hw = xgb.XGBClassifier(
    n_estimators=50,
    max_depth=4,
    learning_rate=0.1,
    tree_method="hist",
    random_state=RANDOM_STATE,
    n_jobs=-1,
    eval_metric="logloss"
)

xgb_norm = xgb.XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.1,
    tree_method="hist",
    random_state=RANDOM_STATE,
    n_jobs=-1,
    eval_metric="logloss"
)

m_hw, _ = eval_grouped(xgb_hw, X_train, y_train, g_train, [])
m_full, _ = eval_grouped(xgb_norm, X_train, y_train, g_train, [])

print(f"XGBoost full (300 trees, depth 6): macro_f1={m_full['macro_f1']:.4f}")
print(f"XGBoost HW   (50 trees,  depth 4): macro_f1={m_hw['macro_f1']:.4f}")
print(f"F1 cost of hardware constraint: {m_full['macro_f1'] - m_hw['macro_f1']:.4f}")
print(f"LR balanced on 8 features:       macro_f1={m['macro_f1']:.4f}")
print(f"Floor:                            macro_f1=0.4168")

XGBoost full (300 trees, depth 6): macro_f1=0.5150
XGBoost HW   (50 trees,  depth 4): macro_f1=0.4299
F1 cost of hardware constraint: 0.0852
LR balanced on 8 features:       macro_f1=0.4732
Floor:                            macro_f1=0.4168


**LR vs. XGBoost Analysis**: 
I evaluated gradient boosting at both full and hardware-feasible size against logistic regression on an identical grouped-CV protocol. At sizes that fit the fabric, the tree ensemble underperforms the linear model (0.4299 vs 0.4732), so the accuracy advantage of boosting is not available at deployable scale. I therefore implemented logistic regression.

In [9]:
import shap

# All 28 candidate features, Training files only
X28 = long_train[GENERIC_PMU_28]
y28 = (long_train["marker"] == "Attack").astype(int)
g28 = long_train["source_file"]

cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
rankings = []

for fold, (tr_idx, te_idx) in enumerate(cv.split(X28, y28, g28)):
    X_tr, X_te = X28.iloc[tr_idx], X28.iloc[te_idx]
    y_tr = y28.iloc[tr_idx]

    pol  = fit_inf_policy(X_tr, ["PA:Z"])
    X_tr = apply_inf_policy(X_tr, pol, ["PA:Z"])
    X_te = apply_inf_policy(X_te, pol, ["PA:Z"])

    model = xgb.XGBClassifier(n_estimators=300, max_depth=6, learning_rate=0.1,
                              tree_method="hist", random_state=RANDOM_STATE,
                              n_jobs=-1, eval_metric="logloss").fit(X_tr, y_tr)

    rng  = np.random.default_rng(RANDOM_STATE)
    samp = X_te.iloc[rng.choice(len(X_te), min(20_000, len(X_te)), replace=False)]
    c    = model.get_booster().predict(xgb.DMatrix(samp), pred_contribs=True)
    rank = pd.Series(np.abs(c[:, :-1]).mean(axis=0), index=samp.columns)
    rank = rank[~rank.index.str.endswith("__open")].sort_values(ascending=False)
    rankings.append(rank)
    print(f"fold {fold}: {list(rank.head(8).index)}")


fold 0: ['PM2:V', 'PM1:V', 'PM5:I', 'F', 'PA3:VH', 'PA:Z', 'PA:ZH', 'PA2:VH']
fold 1: ['PM2:V', 'PM1:V', 'PM5:I', 'PA3:VH', 'F', 'PM7:V', 'PA2:VH', 'PA5:IH']
fold 2: ['PM2:V', 'PM1:V', 'PM5:I', 'F', 'PA3:VH', 'PA:Z', 'PA:ZH', 'PA2:VH']
fold 3: ['PM2:V', 'PM1:V', 'PM5:I', 'F', 'PA7:VH', 'PA3:VH', 'PA:ZH', 'PA5:IH']
fold 4: ['PM2:V', 'PM1:V', 'PM5:I', 'F', 'PA3:VH', 'PA:Z', 'PA2:VH', 'PM4:I']


In [10]:
ranks = pd.DataFrame({f"f{i}": r.rank(ascending=False) for i, r in enumerate(rankings)})
ranks["mean_rank"]  = ranks.mean(axis=1)
ranks["worst_rank"] = ranks.max(axis=1)
display(ranks.sort_values("mean_rank").round(2).head(12))

for f in MINIMAL:
    hits = sum(f in r.head(8).index for r in rankings)
    print(f"{f:10s} top-8 in {hits}/5 folds")

,f0,f1,f2,f3,f4,mean_rank,worst_rank
PM2:V,1.0,1.0,1.0,1.0,1.0,1.0,1.0
PM1:V,2.0,2.0,2.0,2.0,2.0,2.0,2.0
PM5:I,3.0,3.0,3.0,3.0,3.0,3.0,3.0
F,4.0,5.0,4.0,4.0,4.0,4.2,5.0
PA3:VH,5.0,4.0,5.0,6.0,5.0,5.0,6.0
PA:Z,6.0,10.0,6.0,10.0,6.0,7.6,10.0
PA:ZH,7.0,9.0,7.0,7.0,9.0,7.8,9.0
PA2:VH,8.0,7.0,8.0,11.0,7.0,8.2,11.0
PA5:IH,10.0,8.0,11.0,8.0,10.0,9.4,11.0
PM4:I,11.0,11.0,9.0,9.0,8.0,9.6,11.0


PM2:V      top-8 in 5/5 folds
PM1:V      top-8 in 5/5 folds
PM5:I      top-8 in 5/5 folds
F          top-8 in 5/5 folds
PA3:VH     top-8 in 5/5 folds
PM4:I      top-8 in 1/5 folds
PA2:VH     top-8 in 4/5 folds
PA:ZH      top-8 in 3/5 folds


In [11]:
# The stability-ordered feature list: best mean rank first, all 28 of them
ORDER = list(ranks.sort_values("mean_rank").index)

def lr_reduction_curve(X_src, y, groups, order, ks):
    rows = []
    for k in ks:
        feats = order[:k]                                  # take the top k
        z = ["PA:Z"] if "PA:Z" in feats else []            # inf policy only if PA:Z made the cut
        m, _ = eval_grouped(lr_8, X_src[feats], y, groups, z)
        rows.append({"k": k, "macro_f1": m["macro_f1"]})
        print(f"k={k:3d}  macro_f1={m['macro_f1']:.4f}   {feats}")
    return pd.DataFrame(rows)

KS = [1, 2, 3, 5, 8, 12, 20, 28]
lr_curve = lr_reduction_curve(long_train[GENERIC_PMU_28], y28, g28, ORDER, KS)

TOL  = 0.01
full = lr_curve.macro_f1.iloc[-1]                          # score using all 28
knee = int(lr_curve.loc[lr_curve.macro_f1 >= full * (1 - TOL), "k"].min())
print(f"\nall 28: {full:.4f}   knee (within 1%): k={knee}")
print(f"minimal set: {ORDER[:knee]}")

k=  1  macro_f1=0.3917   ['PM2:V']
k=  2  macro_f1=0.2624   ['PM2:V', 'PM1:V']
k=  3  macro_f1=0.4710   ['PM2:V', 'PM1:V', 'PM5:I']
k=  5  macro_f1=0.4757   ['PM2:V', 'PM1:V', 'PM5:I', 'F', 'PA3:VH']
k=  8  macro_f1=0.4808   ['PM2:V', 'PM1:V', 'PM5:I', 'F', 'PA3:VH', 'PA:Z', 'PA:ZH', 'PA2:VH']
k= 12  macro_f1=0.4805   ['PM2:V', 'PM1:V', 'PM5:I', 'F', 'PA3:VH', 'PA:Z', 'PA:ZH', 'PA2:VH', 'PA5:IH', 'PM4:I', 'PA7:VH', 'PA1:VH']
k= 20  macro_f1=0.4817   ['PM2:V', 'PM1:V', 'PM5:I', 'F', 'PA3:VH', 'PA:Z', 'PA:ZH', 'PA2:VH', 'PA5:IH', 'PM4:I', 'PA7:VH', 'PA1:VH', 'PM7:V', 'PM11:I', 'PM3:V', 'PM6:I', 'PM10:I', 'PA10:IH', 'PA6:IH', 'PM12:I']
k= 28  macro_f1=0.4839   ['PM2:V', 'PM1:V', 'PM5:I', 'F', 'PA3:VH', 'PA:Z', 'PA:ZH', 'PA2:VH', 'PA5:IH', 'PM4:I', 'PA7:VH', 'PA1:VH', 'PM7:V', 'PM11:I', 'PM3:V', 'PM6:I', 'PM10:I', 'PA10:IH', 'PA6:IH', 'PM12:I', 'PA12:IH', 'PA4:IH', 'DF', 'PA11:IH', 'PM8:V', 'PM9:V', 'PA8:VH', 'PA9:VH']

all 28: 0.4839   knee (within 1%): k=8
minimal set: ['PM2:V', 'PM1:V',

## Locked configuration

Everything below this point is fixed before the holdout is scored.

Model inputs (9, in this column order):
PM2:V, PM1:V, PM5:I, F, PA3:VH, PA:Z, PA:ZH, PA2:VH, PA:Z__open

The 8 features are the k=8 knee of the logistic regression reduction curve over the
stability-averaged SHAP ranking, derived on the 12 training files only. PA:Z__open is
appended by the infinity policy because PA:Z is in the set.

Combiner: OR across the 4 relays. A snapshot is labeled Attack if at least 1 of the 4
relay classifiers outputs Attack. The vote threshold was swept over 1, 2, 3, and 4 on
training data and 1 was selected.

Holdout: data6.csv, data11.csv, data15.csv. Held out by source file. Never used for
feature selection, model selection, or threshold selection.

The holdout is scored once. The OR result is the reported headline regardless of whether
another combiner scores higher on these three files.

In [12]:
from sklearn.base import clone

FEATURES = ORDER[:knee]         

def oof_scores(model, frame, feats, groups, n_splits=5):
    #One out-of-fold decision score per relay-row
    X = frame[feats]
    y = (frame["marker"] == "Attack").astype(int)
    z = ["PA:Z"] if "PA:Z" in feats else []
    cv = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_STATE)
    g  = pd.Series(groups).reset_index(drop=True)

    scores = np.empty(len(y), dtype=float)
    for tr_idx, te_idx in cv.split(X, y, g):
        assert not set(g.iloc[tr_idx]) & set(g.iloc[te_idx]), "group leak"
        pol    = fit_inf_policy(X.iloc[tr_idx], z)
        fitted = clone(model).fit(apply_inf_policy(X.iloc[tr_idx], pol, z), y.iloc[tr_idx])
        scores[te_idx] = fitted.decision_function(apply_inf_policy(X.iloc[te_idx], pol, z))
    return scores

long_train["score"] = oof_scores(lr_8, long_train, FEATURES, g_train)
long_train["pred"]  = (long_train["score"] >= 0).astype(int)
print(long_train[["orig_row", "relay_id", "score", "pred", "marker"]].head(8).to_string())

   orig_row  relay_id     score  pred   marker
0         0         1 -0.222768     0  Natural
1         1         1 -0.152379     0  Natural
2         2         1 -0.152394     0  Natural
3         3         1 -0.153232     0  Natural
4         4         1 -0.156565     0  Natural
5         5         1 -0.157608     0  Natural
6         6         1 -0.157288     0  Natural
7         7         1 -0.158171     0  Natural


In [13]:
# Each snapshot (orig_row) has exactly one row per relay -> pivot to 4 columns
wide_scores = long_train.pivot(index="orig_row", columns="relay_id", values="score")
wide_preds  = long_train.pivot(index="orig_row", columns="relay_id", values="pred")

assert wide_scores.shape[1] == 4, "expected 4 relays per snapshot"
assert not wide_scores.isna().any().any(), "some snapshot is missing a relay"

# Snapshot label: all four rows share it, so take the first
snap_y = (long_train.groupby("orig_row")["marker"].first() == "Attack").astype(int)
snap_y = snap_y.loc[wide_scores.index]          # align to the pivot's row order

n_attack = wide_preds.sum(axis=1)               # how many of the 4 said Attack

combos = {
    "majority vote (tie -> Attack)":  (n_attack >= 2).astype(int),
    "majority vote (tie -> Natural)": (n_attack >= 3).astype(int),
    "mean of scores":                 (wide_scores.mean(axis=1) >= 0).astype(int),
    "unanimous Attack":               (n_attack == 4).astype(int),
    "any relay says Attack":          (n_attack >= 1).astype(int),
}

rows = []
for r in wide_preds.columns:                    # each relay judged alone, at snapshot level
    rows.append({"method": f"relay R{r} alone", **_metrics(snap_y, wide_preds[r])})
for name, pred in combos.items():
    rows.append({"method": name, **_metrics(snap_y, pred)})
rows.append({"method": "always Attack (floor)",
             **_metrics(snap_y, np.ones(len(snap_y), dtype=int))})

res = pd.DataFrame(rows).set_index("method")[["accuracy", "precision", "recall", "macro_f1"]]
display(res.round(4).sort_values("macro_f1", ascending=False))
print(f"\nsnapshots: {len(snap_y):,}   attack rate: {snap_y.mean():.4f}"
      f"   floor p/(1+p): {snap_y.mean()/(1+snap_y.mean()):.4f}")

,accuracy,precision,recall,macro_f1
method,,,,
any relay says Attack,0.5604,0.7347,0.5992,0.5191
mean of scores,0.5136,0.7348,0.4962,0.4947
majority vote (tie -> Attack),0.5193,0.7268,0.5209,0.4942
relay R4 alone,0.5153,0.7298,0.5073,0.4935
relay R1 alone,0.5115,0.7267,0.5036,0.4899
relay R2 alone,0.4792,0.7284,0.4287,0.4701
relay R3 alone,0.4773,0.7234,0.4310,0.4674
majority vote (tie -> Natural),0.4586,0.7215,0.3908,0.4534
unanimous Attack,0.4450,0.7214,0.3597,0.4425



snapshots: 62,883   attack rate: 0.7123   floor p/(1+p): 0.4160


In [14]:
# What if one model saw all four relays' features at once, instead of 4 numbers?
cent = long_train.pivot(index="orig_row", columns="relay_id", values=FEATURES)
cent.columns = [f"R{r}-{f}" for f, r in cent.columns]
cent = cent.loc[wide_scores.index]

snap_g = long_train.groupby("orig_row")["source_file"].first().loc[cent.index]
z_cent = [c for c in cent.columns if c.endswith("PA:Z")]

m_cent, _ = eval_grouped(lr_8, cent, snap_y, snap_g, z_cent)
print(f"centralized ({cent.shape[1]} features, all relays): macro_f1={m_cent['macro_f1']:.4f}")


centralized (32 features, all relays): macro_f1=0.5642


In [15]:
#Cell 18: Holdout Test Set Voting Evaluation
# Fit final model on ALL training files, score the 3 locked holdout files,
# and apply the same combiners to get the definitive hardware-target numbers.

# Step 1: Fit inf policy on training data only (PA:Z is in FEATURES, so inf policy must be applied)
pol_final = fit_inf_policy(long_train[FEATURES], ["PA:Z"])

X_train_f = apply_inf_policy(long_train[FEATURES], pol_final, ["PA:Z"])
X_test_f  = apply_inf_policy(long_test[FEATURES],  pol_final, ["PA:Z"])
y_train_f = (long_train["marker"] == "Attack").astype(int)

# Step 2: Train one final model on the full training set
final_lr = clone(lr_8).fit(X_train_f, y_train_f)

# Step 3: Score the holdout relay-rows
long_test = long_test.copy()
long_test["score"] = final_lr.decision_function(X_test_f)
long_test["pred"]  = (long_test["score"] >= 0).astype(int)

# Step 4: Pivot to wide (one row per snapshot, one col per relay)
wide_scores_t = long_test.pivot(index="orig_row", columns="relay_id", values="score")
wide_preds_t  = long_test.pivot(index="orig_row", columns="relay_id", values="pred")

assert wide_scores_t.shape[1] == 4, "expected 4 relays per snapshot"
assert not wide_scores_t.isna().any().any(), "missing relay in test set"

snap_y_t = (long_test.groupby("orig_row")["marker"].first() == "Attack").astype(int)
snap_y_t  = snap_y_t.loc[wide_scores_t.index]

# Step 5: Same combiners as OOF analysis
n_attack_t = wide_preds_t.sum(axis=1)
combos_t = {
    "any relay says Attack":           (n_attack_t >= 1).astype(int),
    "majority vote (tie -> Attack)":   (n_attack_t >= 2).astype(int),
    "majority vote (tie -> Natural)":  (n_attack_t >= 3).astype(int),
    "mean of scores":                  (wide_scores_t.mean(axis=1) >= 0).astype(int),
    "unanimous Attack":                (n_attack_t == 4).astype(int),
}

rows_t = []
for r in wide_preds_t.columns:
    rows_t.append({"method": f"relay R{r} alone", **_metrics(snap_y_t, wide_preds_t[r])})
for name, pred in combos_t.items():
    rows_t.append({"method": name, **_metrics(snap_y_t, pred)})
rows_t.append({"method": "always Attack (floor)",
               **_metrics(snap_y_t, np.ones(len(snap_y_t), dtype=int))})

res_t = pd.DataFrame(rows_t).set_index("method")[["accuracy","precision","recall","macro_f1"]]
display(res_t.round(4).sort_values("macro_f1", ascending=False))
print(f"\nHoldout files: {TEST_FILES}")
print(f"Holdout snapshots: {len(snap_y_t):,}   attack rate: {snap_y_t.mean():.4f}")

# Step 6: Print the headline narrative numbers
any_attack_f1 = res_t.loc["any relay says Attack", "macro_f1"]
floor_f1      = res_t.loc["always Attack (floor)", "macro_f1"]

print(f"\n── Headline numbers ──────────────────────────────────────")
print(f"Floor (always-Attack):          {floor_f1:.4f}")
print(f"Best combiner (any-Attack):     {any_attack_f1:.4f}")

,accuracy,precision,recall,macro_f1
method,,,,
any relay says Attack,0.5614,0.7242,0.6058,0.5214
mean of scores,0.5277,0.7303,0.5185,0.5080
majority vote (tie -> Attack),0.5201,0.7158,0.5243,0.4966
relay R4 alone,0.5145,0.7176,0.5081,0.4945
relay R1 alone,0.5128,0.7147,0.5090,0.4921
relay R3 alone,0.4950,0.7230,0.4547,0.4845
relay R2 alone,0.4932,0.7214,0.4525,0.4827
majority vote (tie -> Natural),0.4733,0.7171,0.4123,0.4674
unanimous Attack,0.4606,0.7173,0.3820,0.4575



Holdout files: ['data6.csv', 'data11.csv', 'data15.csv']
Holdout snapshots: 15,494   attack rate: 0.7018

── Headline numbers ──────────────────────────────────────
Floor (always-Attack):          0.4124
Best combiner (any-Attack):     0.5214


In [16]:
# Centralized ceiling, evaluated on the SAME holdout files as the cell above.
# One model sees all 4 relays' features at once, instead of 4 one-bit votes.

# Pivot both frames: one row per snapshot, 4 relays x 8 features = 32 columns
cent_train = long_train.pivot(index="orig_row", columns="relay_id", values=FEATURES)
cent_train.columns = [f"R{r}-{f}" for f, r in cent_train.columns]

cent_test = long_test.pivot(index="orig_row", columns="relay_id", values=FEATURES)
cent_test.columns = [f"R{r}-{f}" for f, r in cent_test.columns]
cent_test = cent_test[cent_train.columns]        # force identical column order

# PA:Z appears once per relay, so the inf policy covers 4 columns here, not 1
z_cent = [c for c in cent_train.columns if c.endswith("PA:Z")]
pol_cent = fit_inf_policy(cent_train, z_cent)              # training snapshots only
cent_train_f = apply_inf_policy(cent_train, pol_cent, z_cent)
cent_test_f  = apply_inf_policy(cent_test,  pol_cent, z_cent)

# Snapshot labels for the training side
snap_y_train = (long_train.groupby("orig_row")["marker"].first() == "Attack").astype(int)
snap_y_train = snap_y_train.loc[cent_train.index]

cent_test_f = cent_test_f.loc[snap_y_t.index]              # align to cell 18's row order

cent_model  = clone(lr_8).fit(cent_train_f, snap_y_train)
m_cent_hold = _metrics(snap_y_t, cent_model.predict(cent_test_f))

cent_f1_hold = m_cent_hold["macro_f1"]
recovery = (any_attack_f1 - floor_f1) / (cent_f1_hold - floor_f1) * 100

print(f"centralized inputs: {cent_test_f.shape[1]} ({len(z_cent)*0 + 32} features + {len(z_cent)} open flags)")
print(f"\nHoldout, all three numbers from data6/11/15:")
print(f"  floor (always-Attack):   {floor_f1:.4f}")
print(f"  OR consensus (4 bits):   {any_attack_f1:.4f}")
print(f"  centralized (all feats): {cent_f1_hold:.4f}")
print(f"  recovery above floor:    {recovery:.1f}%")


centralized inputs: 36 (32 features + 4 open flags)

Holdout, all three numbers from data6/11/15:
  floor (always-Attack):   0.4124
  OR consensus (4 bits):   0.5214
  centralized (all feats): 0.5724
  recovery above floor:    68.1%


In [17]:
scaler = final_lr.named_steps["scale"]
clf = final_lr.named_steps["clf"]

#get the correct order of the columns
INPUT_COLS = list(X_train_f.columns)
assert len(INPUT_COLS) == 9, INPUT_COLS

np.savez(
    "../models/lr_weights.npz",
    feature_names = np.array(INPUT_COLS), 
    coef = clf.coef_[0],
    intercept = np.array(clf.intercept_[0]),
    scaler_mean = scaler.mean_,
    scaler_scale = scaler.scale_,
    paz_cap = np.array(pol_final["PA:Z"]),
    vote_threshold = np.array(1),
)

for name, w, mu, sd in zip(INPUT_COLS, clf.coef_[0], scaler.mean_, scaler.scale_):
    print(f"{name:14s} w={w:+.6f}  mean={mu:+.4e}  scale = {sd:.4e}")
print(f"\nIntercept: {clf.intercept_[0]:+.6f}")
print(f"PA:Z cap: {pol_final['PA:Z']:.4f}") 


PM2:V          w=+0.139734  mean=+1.2950e+05  scale = 9.6764e+03
PM1:V          w=-0.276417  mean=+1.2890e+05  scale = 1.3170e+04
PM5:I          w=-0.068755  mean=+3.8914e+02  scale = 1.5296e+02
F              w=-0.034232  mean=+5.9991e+01  scale = 7.0201e-01
PA3:VH         w=+0.045464  mean=+5.6616e+00  scale = 9.7292e+01
PA:Z           w=+0.280738  mean=+1.3339e+01  scale = 1.4993e+01
PA:ZH          w=+0.022247  mean=-1.0837e+00  scale = 1.8157e+00
PA2:VH         w=-0.001808  mean=+3.9146e+00  scale = 1.1128e+02
PA:Z__open     w=-0.354352  mean=+3.4930e-02  scale = 1.8360e-01

Intercept: +0.011363
PA:Z cap: 85.2800


In [20]:
z_test = scaler.transform(X_test_f)

np.savez(
    "../models/holdout_vectors.npz",
    feature_names  = np.array(INPUT_COLS),
    X_capped = X_test_f.to_numpy(dtype=np.float32),
    X_std = z_test.astype(np.float32),                               
    score = long_test["score"].to_numpy(),
    relay_bit = long_test["pred"].to_numpy(dtype=np.int8),
    orig_row = long_test["orig_row"].to_numpy(),
    relay_id = long_test["relay_id"].to_numpy(),
    snapshot_id = snap_y_t.index.to_numpy(),
    snapshot_label = snap_y_t.to_numpy(dtype=np.int8),
    snapshot_or = combos_t["any relay says Attack"].to_numpy(dtype=np.int8),
)

# Verify the saved weights actually reproduce sklearn. This is the seed of lr_inference.py.
w = np.load("../models/lr_weights.npz")
manual = (X_test_f.to_numpy() - w["scaler_mean"]) / w["scaler_scale"]
manual_score = manual @ w["coef"] + w["intercept"]

err = np.abs(manual_score - long_test["score"].to_numpy()).max()
assert np.allclose(manual_score, long_test["score"].to_numpy()), f"weights do not reproduce model, max err {err}"
print(f"saved weights reproduce sklearn, max score error: {err:.3e}")
print(f"holdout rows: {len(long_test):,}   snapshots: {len(snap_y_t):,}")

saved weights reproduce sklearn, max score error: 0.000e+00
holdout rows: 61,976   snapshots: 15,494
